## Imports

In [254]:
import numpy as np
import scipy
import sklearn
import pandas as pd
import matplotlib as mpl
from matplotlib import pyplot as pyplot
import json
import os
import copy
from numpy import random as rng

from misc import move

In [255]:
rng = np.random.default_rng()

## Load the data to visualize it

In [256]:
all_data_matrix = pd.read_csv('../Data/raw_data/all_subjects.csv')

In [257]:
all_data_matrix = all_data_matrix.drop('t',axis=1)
all_data_matrix = all_data_matrix[all_data_matrix['event'] != 'BONUS_FAIL']
all_data_matrix = all_data_matrix[all_data_matrix['event'] != 'BONUS_SUCCESS']

In [258]:
subject_1 = all_data_matrix[all_data_matrix['subject']=='A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB']

In [259]:
subject_1_prb10206_7 = subject_1[subject_1['instance']=='prb10206_7']

In [260]:
subject_1_prb10206_7

,subject,event,move,instance,piece,target
5180,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,start,0,prb10206_7,-1,-1
5181,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_start,0,prb10206_7,4,2
5182,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_end,0,prb10206_7,4,3
5183,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_start,1,prb10206_7,3,8
5184,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_end,1,prb10206_7,3,2
...,...,...,...,...,...,...
17231,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_end,10,prb10206_7,8,16
17232,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_start,11,prb10206_7,8,16
17233,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,win,11,prb10206_7,8,16
17234,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_end,11,prb10206_7,8,16


In [262]:
def parse_data_matrix(df):
    df = copy.deepcopy(df)
    df['sq_change'] = df.shift(-1)['target'] - df['target']
    df['dist'] = (df['sq_change'])*((df['sq_change'] % 6) != 0) + (df['sq_change'] // 6)*((df['sq_change'] % 6) == 0)
    df['piece'] = df['piece'].where(df['piece'] != 8, -1)
    df['piece'] = df['piece'].where(df['piece'] == -1, df['piece'] + 1)
    df.loc[df['event'].isin(pd.Series(['win','start','restart','surrender'])), ['piece','target','sq_change','dist']] = None
    df['piece'] = df['piece'].astype('Int64')
    df['p_as_str'] = df['piece'].astype('str')
    df = df[df['event'] != 'drag_end']
    df = df[~((df['event'] == 'win') & (df['event'].shift() == 'win'))]
    df = df[df['dist'] != 0]
    df['dist'] = df['dist'].astype('Int64')
    df.loc[df['event'] == 'drag_start','event'] = 'move'
    df = df.drop(columns=['target','sq_change','piece'])
    df = df.fillna(0)
    df['solve_instance'] = ((df['event'] == 'start') & (df['event'].shift().isin(pd.Series(['win','surrender'])))).cumsum()
    tot_solve_insts = pd.unique(df['solve_instance'])
    for solve_instance in tot_solve_insts:
        
        pass
    return df

In [263]:
s1_prb10206_7_processed = parse_data_matrix(s1_prb10206_7)
s1_prb10206_7_processed.to_csv('../Data/my_processed_data/s1_prb10206_7_processed.csv')

In [278]:
len(all_data_matrix)

279655

In [282]:
subject_1_p = parse_data_matrix(subject_1)
first_g_first_instc = subject_1_p[subject_1_p['instance']=='prb55384_14']
fgfi_no_dup = first_g_first_instc[first_g_first_instc['solve_instance']==pd.unique(first_g_first_instc['solve_instance'])[0]]
fgfi_no_dup

,subject,event,move,instance,dist,p_as_str,solve_instance
0,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,start,0,prb55384_14,0,0,0
1,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,0,prb55384_14,1,3,0
5,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,1,prb55384_14,-2,3,0
15,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,2,prb55384_14,2,3,0
21,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,3,prb55384_14,1,-1,0
27,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,4,prb55384_14,4,8,0
29,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,5,prb55384_14,-1,2,0
31,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,6,prb55384_14,-1,7,0
33,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,7,prb55384_14,-1,6,0
43,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,8,prb55384_14,-1,-1,0


In [265]:
shift = 6014
f1 = all_data_matrix.reset_index().loc[(0+shift):(6012+shift)].reset_index().drop(columns='index')
f2 = all_data_matrix.reset_index().loc[(6013+shift):(6013+6012+shift)].reset_index().drop(columns='index')
(f1 == f2).all()

level_0     False
subject     False
event        True
move         True
instance    False
piece        True
target       True
dtype: bool

In [266]:
all_data_matrix.reset_index().loc[6010:6016]

,index,subject,event,move,instance,piece,target
6010,6010,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,win,6,prb13171_7,8,13
6011,6011,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_end,6,prb13171_7,8,16
6012,6012,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,win,7,prb13171_7,-1,-1
6013,6016,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,start,0,prb55384_14,-1,-1
6014,6017,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_start,0,prb55384_14,2,14
6015,6018,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_end,0,prb55384_14,2,20
6016,6019,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_start,1,prb55384_14,2,20


In [267]:
processed_data = parse_data_matrix(all_data_matrix)

In [268]:
processed_data_head = processed_data.head(200)
processed_data_head

,subject,event,move,instance,dist,p_as_str,solve_instance
0,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,start,0,prb55384_14,0,0,0
1,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,0,prb55384_14,1,3,0
5,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,1,prb55384_14,-2,3,0
15,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,2,prb55384_14,2,3,0
21,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,3,prb55384_14,1,-1,0
...,...,...,...,...,...,...,...
534,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,11,prb23404_14,3,6,4
537,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,win,12,prb23404_14,0,0,4
540,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,start,0,prb47495_14,0,0,5
541,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,0,prb47495_14,-1,2,5


In [269]:
processed_data.to_csv('../Data/my_processed_data/processed_data.csv')